# Notebook: BCI_23_CarDet_Crit_Entrada_Ant
*********************************************************************************

## Informacion del Notebook

### Encabezado
**************************************************************************
* Nombre: BCI_23_CarDet_Crit_Entrada_Ant.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/2997520011902103
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 12/08/2022
* Descripcion: obtiene todos las operaciones deterioradas del periodo anterior
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 10/02/2025 
* Descripción: Se agrega nueva regla LIR - No se condirera los clientes que no vienen informado en el archivo en el periodo actual.      
***************************************************************************

**************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 09/04/2025 
* Descripción: Se elimina la nueva regla LIR - Se mantiene la logica de que en caso de entrar a LIR el cliente no sale.      
***************************************************************************

### Tablas Entrada y Salida
**************************************************************************
#### Tablas Entrada: 
* {base_silver_x}.tbl_cd_d00_segmentado_pant
***************************************************************************
#### Tablas Salida: 
* {base_silver_x}.tbl_cd_cartdet_crit_ent_crit
***************************************************************************


## Carga Dependencias

### Carga funciones comunes

In [0]:
%run "./Funciones_Comunes"

# Notebook: Funciones_Comunes
**************************************************************************

## Informacion del Notebook 

### Encabezado
**************************************************************************
* Nombre: Funciones_Comunes.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/3570959530595695
* Autor: Gabriel Martínez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 23/09/2023
* Descripcion: Notebook con funciones genéricas que pueden ser usadas por otros notebooks.
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 15/02/2025 
* Descripción: Se cambio el metodo de cancelacion utilizando el comando (raise) y se incorporada la funcion de ir a buscar el ultimo dia calendario. Tambien se agrego una nueva funcion (obtener_estados_tablas).  
***************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 25/04/2025 
* Descripción: Se modifico la funcion extension_archivos para que cuando la vigencia sea previa, asigne extencion .PRV.  
***************************************************************************
#### Mantención Nro: 3
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 08/07/2025 
* Descripción: Se realiza una mejora en la funcion mostrar_variacion_criterio.  
***************************************************************************

## Carga librerias

...

### obtener archivo periodo anterior

### concatena archivos

###primer_dia_mes_sig

###Calcula fecha X meses atras

## FIN definición de funciones

## Parámetros

### Setea Parámetros

In [0]:

dbutils.widgets.text("fecha_w","","01-Fecha:")
dbutils.widgets.text("bd_silver_w","","03-Nombre BD Silver:")

fecha_x = dbutils.widgets.get("fecha_w") 
base_silver_x = dbutils.widgets.get("bd_silver_w")

spark.conf.set("bci.fecha", fecha_x)
spark.conf.set("bci.dbnamesilver", base_silver_x)

print(f"Fecha de Proceso actual: [fecha_x] {fecha_x}")
print(f"Nombre BD Silver: [base_silver_x] {base_silver_x}")


Fecha de Proceso actual: [fecha_x] 20250930
Nombre BD Silver: [base_silver_x] dsr_gld_bciwork_db


### Valida parámetros

In [0]:
valida_parametro(fecha_x)

In [0]:
valida_parametro(base_silver_x)

## INICIO Proceso extraccion y transformacion
--------------------------------------
- Por cada fuente que se utilice se debe:
     - Titulo: generar un titulo generico, con nombre fuente, y descripcion del proposito de la extraccion
     - Extraer: para el periodo, o rango de fecha que se necesita la iformacion. Debe tener el prefijo tmp_EXT_{nombrefuente}
     - Transformar: generar la informacion necesaria para la salida final. Se pueden generar mas de una tabla temporal para llegar al resultado final. Debe tener el prefijo tmp_RES_{nombre}_correlativo


### Parametrizacion
---
* define y asigna valores a los parametros


In [0]:
p_ind_cartdet='D'
p_crit_det = 1,7,8,9,10,11,12,13
p_periodo_evaluacion='p_anterior'

print(f"p_ind_cartdet: {p_ind_cartdet}")
print(f"p_crit_det: {p_crit_det}")
print(f"p_periodo_evaluacion: {p_periodo_evaluacion}")


p_ind_cartdet: D
p_crit_det: (1, 7, 8, 9, 10, 11, 12, 13)
p_periodo_evaluacion: p_anterior


In [0]:
# Calcula periodo en base a la fecha
periodo_x=fecha_x[:6]

print(f"Periodo: [periodo_x] {periodo_x}")

Periodo: [periodo_x] 202509


### Deterioro operaciones periodo anterior
--------------------------------------
- obtiene todas los deterioros del cliente informados en periodo anterior


In [0]:
paso_query20 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_1 as
SELECT
        A.periodo_cierre          AS periodo_cierre,
        A.fecha_cierre            AS fecha_cierre,
        A.tipo_proceso            AS tipo_proceso,
        A.rut_cliente             AS rut_cliente,
        A.dv_rut_cliente          AS dv_rut_cliente,
        A.tipo_operacion          AS tipo_operacion,
        A.operacion               AS operacion,
        A.sistema                 AS sistema,
        A.segmento                AS segmento,
        A.criterio_entrada        AS criterio_entrada,
        A.origen_deterioro        AS origen_deterioro,
        A.fecha_entrada           AS fecha_entrada,
        CASE 
            WHEN A.criterio_entrada = 1  THEN 'BCI_Individual' 
            WHEN A.criterio_entrada = 12 THEN 'SSFF'
            WHEN A.criterio_entrada = 13 THEN 'Factoring'
            ELSE 'BCI_Grupal'
        END                       AS grupo
FROM 
    {base_silver_x}.tbl_cd_d00_segmentado_pant   A
WHERE
    A.ind_cartdet='{p_ind_cartdet}' 
AND IFNULL(A.criterio_entrada,0)>0
"""


In [0]:
sql_safe(paso_query20)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_1 as
SELECT
        A.periodo_cierre          AS periodo_cierre,
        A.fecha_cierre            AS fecha_cierre,
        A.tipo_proceso            AS tipo_proceso,
        A.rut_cliente             AS rut_cliente,
        A.dv_rut_cliente          AS dv_rut_cliente,
        A.tipo_operacion          AS tipo_operacion,
        A.operacion               AS operacion,
        A.sistema                 AS sistema,
        A.segmento                AS segmento,
        A.criterio_entrada        AS criterio_entrada,
        A.origen_deterioro        AS origen_deterioro,
        A.fecha_entrada           AS fecha_entrada,
        CASE 
            WHEN A.criterio_entrada = 1  THEN 'BCI_Individual' 
            WHEN A.criterio_entrada = 12 THEN 'SSFF'
            WHEN A.criterio_entrada = 13 THEN 'Factoring'
            ELSE 'BCI_Grupal'
        END                       AS grupo
FROM 
    dsr_gld_bciwork_db.tbl_

DataFrame[]

### Salida Temporal a Nivel de Campo Evaludado (tmp_tbl_cartdet_crit_ent_crit)
------------------
* generar salida temporal a nivel de campo evaluado. 
* se registran todas las operaciones evaluadas


In [0]:

paso_query250 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_ent_crit AS
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_1  
"""  

In [0]:
sql_safe(paso_query250)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_ent_crit AS
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_1  



DataFrame[]

## Carga Tablas de Salidas
--------------------------------------
* carga resultados a tablas de salidas del notebook

### Carga Tabla Evaluacion 


#### Reproceso (Elimina registros en caso de reprocesos). Tabla no es historica

In [0]:
paso_query300 = f"""DELETE FROM {base_silver_x}.tbl_cd_cartdet_crit_ent_crit where criterio_entrada in {p_crit_det} and periodo_evaluacion = '{p_periodo_evaluacion}' """

In [0]:
sql_safe(paso_query300)

sql_safe: query -> DELETE FROM dsr_gld_bciwork_db.tbl_cd_cartdet_crit_ent_crit where criterio_entrada in (1, 7, 8, 9, 10, 11, 12, 13) and periodo_evaluacion = 'p_anterior' 


DataFrame[num_affected_rows: bigint]

#### Inserta Registros tabla salida

In [0]:
paso_query310 = f"""
INSERT INTO {base_silver_x}.tbl_cd_cartdet_crit_ent_crit
SELECT 
    IFNULL({periodo_x},190001),
    IFNULL({fecha_x},19000101),
    IFNULL(tipo_proceso,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(segmento,' '),
    IFNULL(criterio_entrada,0),
    IFNULL(origen_deterioro,0),
    IFNULL(fecha_entrada,19000101),
    IFNULL(grupo,' '),
    IFNULL('{p_periodo_evaluacion}',' ')
FROM
    tmp_tbl_cartdet_crit_ent_crit  
"""  


In [0]:
sql_safe(paso_query310)

sql_safe: query -> 
INSERT INTO dsr_gld_bciwork_db.tbl_cd_cartdet_crit_ent_crit
SELECT 
    IFNULL(202509,190001),
    IFNULL(20250930,19000101),
    IFNULL(tipo_proceso,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(segmento,' '),
    IFNULL(criterio_entrada,0),
    IFNULL(origen_deterioro,0),
    IFNULL(fecha_entrada,19000101),
    IFNULL(grupo,' '),
    IFNULL('p_anterior',' ')
FROM
    tmp_tbl_cartdet_crit_ent_crit  



DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
##Estadisticas tabla salida


In [0]:
%sql
SELECT
fecha_cierre,
criterio_entrada,
CASE 
  WHEN criterio_entrada=1 THEN 'CLASIFICACION_DETERIORO'
  WHEN criterio_entrada=7 THEN 'MOROSIDAD_NO_HIPCAE'
  WHEN criterio_entrada=8 THEN 'MOROSIDAD_HIPCAE'
  WHEN criterio_entrada=9 THEN 'RENEGOCIADO'
  WHEN criterio_entrada=10 THEN 'REESTRUCTURACION_FORZOSA'
  WHEN criterio_entrada=11 THEN 'LIR'
  WHEN criterio_entrada=12 THEN 'SSFF'
  WHEN criterio_entrada=13 THEN 'FACTORING'
  ELSE 'NO_IDENTIFICADO'    
END                    AS des_criterio_entrada,
COUNT(1) AS CANT_REG
FROM ${bci.dbnamesilver}.tbl_cd_cartdet_crit_ent_crit
GROUP BY 1,2,3
ORDER BY 1,2,3


fecha_cierre,criterio_entrada,des_criterio_entrada,CANT_REG
20250930,1,CLASIFICACION_DETERIORO,4848
20250930,7,MOROSIDAD_NO_HIPCAE,330320
20250930,8,MOROSIDAD_HIPCAE,104999
20250930,9,RENEGOCIADO,11661
20250930,10,REESTRUCTURACION_FORZOSA,1433
20250930,11,LIR,18068
20250930,12,SSFF,74111
20250930,13,FACTORING,591


## Mensaje termino OK

In [0]:
msgerrorx="OK"
dbutils.notebook.exit("{\"coderror\":0, \"msgerror\":\""+msgerrorx+"\"}")